In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, TimeDistributed

# Qno 1:

In [16]:
sentences = [
    "Ali works at Microsoft in New York",
    "Google is based in California",
    "Elon Musk founded Tesla",
    "Paris is a beautiful city",
    "Amazon has offices in Seattle"]

In [17]:
tags = [
    ["B-PER", "O", "O", "B-ORG", "O", "B-LOC", "I-LOC"],
    ["B-ORG", "O", "O", "O", "B-LOC"],
    ["B-PER", "I-PER", "O", "B-ORG"],
    ["B-LOC", "O", "O", "O", "O"],
    ["B-ORG", "O", "O", "O", "B-LOC"]]

In [18]:
tokenizer = Tokenizer(lower=True, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

In [19]:
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

In [20]:
X = tokenizer.texts_to_sequences(sentences)

max_len = max(len(x) for x in X)
X = pad_sequences(X, maxlen=max_len, padding='post')

In [21]:
tag_set = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag = {i: t for t, i in tag2idx.items()}

In [22]:
y = []
for tag_seq in tags:
    y.append([tag2idx[t] for t in tag_seq])

y = pad_sequences(y, maxlen=max_len, padding='post')
y = np.expand_dims(y, -1)

In [23]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    GRU(64, return_sequences=True),
    TimeDistributed(Dense(len(tag_set), activation='softmax'))
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit(X, y, epochs=30, verbose=1)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.1429 - loss: 1.9465
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.4571 - loss: 1.9304
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.6571 - loss: 1.9143
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.6857 - loss: 1.8978
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6857 - loss: 1.8808
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6857 - loss: 1.8630
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6571 - loss: 1.8442
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6571 - loss: 1.8243
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6571 - loss: 1.8031
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.6571 - loss: 1.7803
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.6571 - loss: 1.7558
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6571 - loss: 1.7295
Ep

In [25]:
def predict_entities(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    preds = model.predict(seq)[0]
    pred_tags = [idx2tag[np.argmax(p)] for p in preds]

    words = sentence.split()

    print("\nEntities Found:")
    for w, t in zip(words, pred_tags):
        if t != "O":
            print(f"{w} → {t}")

In [27]:
test_sentence = "Elon Musk works at Tesla in California"
predict_entities(test_sentence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step

Entities Found:


In [28]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, TimeDistributed

# ---------------------------
# Sample training data
# ---------------------------
sentences = [
    "Aliyah works at Microsoft in New York",
    "Google is based in California",
    "Elon Musk founded Tesla",
    "Paris is a beautiful city",
    "Amazon has offices in Seattle"
]

# BIO tags: O = other, PER = person, LOC = location, ORG = organization
tags = [
    ["B-PER", "O", "O", "B-ORG", "O", "B-LOC", "I-LOC"],
    ["B-ORG", "O", "O", "O", "B-LOC"],
    ["B-PER", "I-PER", "O", "B-ORG"],
    ["B-LOC", "O", "O", "O", "O"],
    ["B-ORG", "O", "O", "O", "B-LOC"]
]

# ---------------------------
# Tokenization
# ---------------------------
tokenizer = Tokenizer(lower=True, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

X = tokenizer.texts_to_sequences(sentences)

max_len = max(len(x) for x in X)
X = pad_sequences(X, maxlen=max_len, padding='post')

# Tag mapping
tag_set = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag = {i: t for t, i in tag2idx.items()}

y = []
for tag_seq in tags:
    y.append([tag2idx[t] for t in tag_seq])

y = pad_sequences(y, maxlen=max_len, padding='post')
y = np.expand_dims(y, -1)

# ---------------------------
# Build GRU model
# ---------------------------
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_len),
    GRU(64, return_sequences=True),
    TimeDistributed(Dense(len(tag_set), activation='softmax'))
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# ---------------------------
# Train model
# ---------------------------
model.fit(X, y, epochs=30, verbose=1)

# ---------------------------
# Prediction function
# ---------------------------
def predict_entities(sentence):
    seq = tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding='post')

    preds = model.predict(seq)[0]
    pred_tags = [idx2tag[np.argmax(p)] for p in preds]

    words = sentence.split()

    print("\nEntities Found:")
    for w, t in zip(words, pred_tags):
        if t != "O":
            print(f"{w} → {t}")

# ---------------------------
# Test
# ---------------------------
test_sentence = "Elon Musk works at Tesla in California"
predict_entities(test_sentence)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.0857 - loss: 1.9476
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.4857 - loss: 1.9283
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.6571 - loss: 1.9089
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6857 - loss: 1.8890
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6857 - loss: 1.8684
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.6857 - loss: 1.8469
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6571 - loss: 1.8240
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6571 - loss: 1.7997
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6571 - loss: 1.7737
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6571 - loss: 1.7457
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6571 - loss: 1.7156
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6571 - loss: 1.6832
Epo